In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
import flexiznam as flz
from cottage_analysis.analysis import (
    spheres,
    find_depth_neurons,
)
from typing import Callable
from cottage_analysis.pipelines import pipeline_utils
from dataclasses import dataclass
from typing import Optional

In [ ]:
from matplotlib import pyplot as plt
import matplotlib as mpl  

In [ ]:
session_name="BRAC9972.2a_S20241021"
project="depth_mismatch_seq"
protocol_base="SpheresPermTubeReward"
photodiode_protocol=5
conflicts="skip"
filter_datasets = {"anatomical_only": 3, "ast_neuropil": False}
exclude_datasets = None

flexilims_session = flz.get_flexilims_session(project_id=project)

_, trials_df_all = spheres.sync_all_recordings(
    session_name=session_name,
    flexilims_session=flexilims_session,
    project=project,
    filter_datasets=filter_datasets,
    exclude_datasets=exclude_datasets,
    conflicts=conflicts,
    recording_type="two_photon",
    protocol_base=protocol_base,
    photodiode_protocol=photodiode_protocol,
    return_volumes=True,
)

In [ ]:
try:
    import torch
except Exception as e:
    raise ImportError(
        "This module requires PyTorch. Install with: pip install torch"
    ) from e



In [ ]:
from tqdm import tqdm
from cottage_analysis.analysis import torch_utils
from cottage_analysis.analysis import torch_fit_gaussian_blob as tfit

from cottage_analysis.analysis.torch_utils import Gaussian2DCholeskyBounds, Gaussian2DBounds, Gaussian1DBounds, GaussianAdditiveBounds, Gaussian2DAngleBounds

# Examine the goodness of fit with torch TRF

In [ ]:
rs_array, of_array, _, response_array, depth_list = tfit.process_rs_of_for_fit(trials_df_all)

In [ ]:
# set up the boundary conditions based on the experimental conditions
PARAM_RANGE = {"rs_min": 0.005, "rs_max": 5, "of_min": 0.03, "of_max": 3000, "log_amplitude_max": 10.}

bounds = torch_utils.format_model_bounds("gaussian_2d", **PARAM_RANGE)
lower, upper = torch_utils.vectorise_bounds(bounds)
print(bounds, flush=True)

# set up the torch device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}", flush=True)
dtype = "float64"

torch_device = torch.device(device)
torch_dtype = torch.float64 if dtype == "float64" else torch.float32

# create tensors for stimuli and responses
X = (torch.as_tensor(rs_array, device=torch_device, dtype=torch_dtype), torch.as_tensor(of_array, device=torch_device, dtype=torch_dtype))
y = torch.as_tensor(response_array, device=torch_device, dtype=torch_dtype)

n_samples, n_rois = y.shape
print(f"n_samples={n_samples}, n_rois={n_rois}", flush=True)

In [ ]:
# Load the neurons_df to choose neurons that are actually depth selective and therefore should have fits that converge
neurons_df_path = "/nemo/lab/znamenskiyp/home/shared/projects/depth_mismatch_seq/BRAC9972.2a/S20241021/neurons_df.pickle"
neurons_df = pd.read_pickle(neurons_df_path)

NICE_ROIS = [495, 519, 853, 1285, 155]

print(f"Production code R^2 for nice ROIs: {np.nanmedian(neurons_df.rsof_rsq_closedloop_crossval_g2d.iloc[NICE_ROIS])}")

In [ ]:
import cottage_analysis.analysis.torch_utils as torch_utils
from cottage_analysis.analysis.torch_fit_gaussian_blob import gaussian_2d_cholesky

In [ ]:
N_STARTS=5
init_params = torch_utils.generate_n_inits_all_rois(
    n_starts=N_STARTS,
    X=X,
    y=y[:, NICE_ROIS],
    model="gaussian_2d",
    bounds=bounds,
    rng_seed=42,
    device=torch_device,
    dtype=torch_dtype,
    apply_bounds=True,
)
init_params = torch_utils.decode_params(init_params, model="gaussian_2d", bounds=bounds)

model_func = gaussian_2d_cholesky

In [ ]:
# Do TRF curve fitting on a good ROI only with N_STARTS
trf_fit = torch_utils.Curve_fit(
    X=X,
    y=y[:, NICE_ROIS],
    params=init_params,
    bounds=bounds,
    model_func=model_func,
    n_starts=N_STARTS,
    n_iters=500,
    method="trf",
)

params_fit = trf_fit.fit()
r2_final = trf_fit.r2.view(len(NICE_ROIS), N_STARTS)
best_r2, best_idx = r2_final.max(dim=1)
best_params = params_fit.view(len(NICE_ROIS), N_STARTS, -1)[torch.arange(len(NICE_ROIS)), best_idx]

print("Best R^2:", best_r2)
print("Scipy R^2:", neurons_df.rsof_rsq_closedloop_crossval_g2d.iloc[NICE_ROIS].values)


# Compare the gaussian blob shape

`log_l11`, `l21`, and `log_l22` are the parameters controlling the shape of the Gaussian blob but can disagree wildly from fit to fit, while the R-sqaured
is identical to 4 decimal places. To compare the shape of the Gaussian blob between the two fits, we can reconstruct the covariance matrix from the fit params and then get the angle
parameterised fit values for comparison.

In [ ]:
MIN_SIGMA = 0.25
from cottage_analysis.analysis.fit_gaussian_blob import (
    _effective_precision_eig,
    get_gaussian_angle,
    get_semimajor_length,
    get_semiminor_length,
)

print(f"Saturation eigenvalue (1/(2*min_sigma)) = {1.0 / (2 * MIN_SIGMA):.4f}\n")
popts = neurons_df.rsof_popt_closedloop_crossval_g2d.iloc[NICE_ROIS].to_list()
popt_torch = best_params.detach().cpu().numpy()

for i in range(5):
    print(f"\nIteration {i+1}")
    this_popt_scipy = popts[i]
    this_popt_torch = popt_torch[i]

    results = {}
    for label, popt in [("scipy", this_popt_scipy), ("torch", this_popt_torch)]:
        eigvals, _ = _effective_precision_eig(popt, min_sigma=MIN_SIGMA)
        angle = get_gaussian_angle(popt, min_sigma=MIN_SIGMA)
        semimajor = get_semimajor_length(popt, min_sigma=MIN_SIGMA)
        semiminor = get_semiminor_length(popt, min_sigma=MIN_SIGMA)
        results[label] = (eigvals, angle, semimajor, semiminor)
        print(
            f"{label:>6s}: effective eigenvalues={eigvals}, angle={angle:.2f} deg, "
            f"semimajor={semimajor:.4f}, semiminor={semiminor:.4f}"
        )

    eig_scipy, angle_scipy = results["scipy"][0], results["scipy"][1]
    eig_torch, angle_torch = results["torch"][0], results["torch"][1]
    eig_diff = np.abs(eig_scipy - eig_torch).max()
    angle_diff = abs(angle_scipy - angle_torch)

    print(f"\nMax abs diff in effective eigenvalues: {eig_diff:.4f}")
    print(f"Angle diff: {angle_diff:.2f} deg")

    if eig_diff < 0.1 and angle_diff < 5:
        print("Effective shape is basically the same.")
    else:
        print("Effective shape differs significantly.")

## Load fits from torch `.parquet` files and compare to the output in `neurons_df.pickle`

In [ ]:
from matplotlib import pyplot as plt
from pathlib import Path

In [ ]:
torch_path = Path(neurons_df_path).parent / "torch"
torch_files = list(torch_path.glob("fit_rs_of_tuning_*.parquet"))

tmp = []
for f in torch_files:
    tmp.append(pd.read_parquet(f))
    
torch_df = tmp[0]
for df in tmp[1:]:
    torch_df = torch_df.merge(df, on="roi", how="outer")

torch_df = torch_df.copy()

In [ ]:

torch_df = torch_df.rename(columns={col: col + "_torch" for col in torch_df.columns if col != "roi"})

In [ ]:
# merge the dataframes
comparison_df = neurons_df.loc[:, ["roi", "rsof_popt_closedloop_crossval_g2d", "rsof_rsq_closedloop_crossval_g2d"]]
comparison_df = comparison_df.merge(torch_df, on="roi", how="left")

In [ ]:
plt.scatter(x = comparison_df["rsof_rsq_closedloop_crossval_g2d"].values, y = comparison_df["rsof_rsq_closedloop_crossval_g2d_torch"].values, alpha=0.5)
x_min = min(comparison_df["rsof_rsq_closedloop_crossval_g2d"].min(), comparison_df["rsof_rsq_closedloop_crossval_g2d_torch"].min())
x_max = max(comparison_df["rsof_rsq_closedloop_crossval_g2d"].max(), comparison_df["rsof_rsq_closedloop_crossval_g2d_torch"].max())
plt.plot([x_min, x_max], [x_min, x_max], "k--")
plt.xlabel("scipy R-squared")
plt.ylabel("torch R-squared")
plt.title("Cholesky-parameterised bivariate gaussian")
plt.show()